# **Testando Gradio**

In [ ]:
# importações

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Carrega as variáveis de ambiente em um arquivo chamado .env
# Imprime os prefixos das chaves para ajudar com qualquer depuração

load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')

if not api_key:
    print("Nenhuma chave de API da OpenRouter foi encontrada - por favor verifique o seu arquivo .env!")
elif not api_key.startswith("sk-or-"):
    print("Uma chave de API foi encontrada, mas não começa com sk-or-; por favor, verifique se você está usando a chave correta da OpenRouter")
else:
    print("Chave de API da OpenRouter encontrada e parece boa até agora!")

In [ ]:
# Inicializar

openai = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)
MODEL = 'nvidia/nemotron-3-super-120b-a12b:free'

In [ ]:
# Mais uma vez, estarei no modo cientista e alterarei essa variável global durante o laboratório

system_message = "Você é um assistente útil"

## E agora, escrevendo um novo callback

Agora precisamos escrever uma função chamada:

`chat(message, history)`

Que será uma função de callback que daremos ao gradio.

### O trabalho desta função

Receber uma mensagem, receber a conversa anterior e retornar a resposta.


In [ ]:
def chat(message, history):
    return "bananas"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    return f"Você disse {message} e o histórico é {history} mas eu ainda digo bananas"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## OK! Vamos escrever um callback de chat um pouco melhor!

In [ ]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## OK, vamos continuar!

Usando uma mensagem do sistema para adicionar contexto e dar um exemplo de resposta... isso é "one shot prompting" novamente

In [ ]:
system_message = "Você é um assistente útil em uma loja de roupas. Você deve tentar encorajar gentilmente \
o cliente a experimentar itens que estão em promoção. Chapéus estão com 60% de desconto e a maioria dos outros itens tem 50% de desconto. \
Por exemplo, se o cliente disser 'Estou procurando um chapéu para comprar', \
você pode responder algo como 'Maravilha - temos muitos chapéus - incluindo vários que fazem parte do nosso evento de vendas.'\
Encoraje o cliente a comprar chapéus se não tiver certeza do que levar."

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
system_message += "\nSe o cliente perguntar por sapatos, você deve responder que os sapatos não estão em promoção hoje, \
mas lembre o cliente de olhar os chapéus!"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'cinto' in message.lower():
        relevant_system_message += " A loja não vende cintos; se for perguntado sobre cintos, certifique-se de apontar outros itens em promoção."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()